In [1]:
import pandas as pd
import numpy as np

import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [2]:
df = pd.read_csv(r"C:\Users\LOQ\OneDrive\Desktop\iot\archive (1)\train_test_network.csv")

In [3]:
# Columns justified by EDA
keep_cols = [
    # core network semantics
    'proto',
    'service',
    'conn_state',

    # numeric behavior (to be transformed later)
    'duration',
    'src_bytes',
    'dst_bytes',
    'src_pkts',
    'dst_pkts',
    'missed_bytes',

    # DNS behavior
    'dns_rejected',

    # HTTP behavior
    'http_method',
    'http_status_code',

    # SSL behavior
    'ssl_established',

    # Weird / anomaly indicators
    'weird_notice',

    # labels
    'label',
    'type'
]

df_work = df[keep_cols].copy()

print("Working dataframe shape:", df_work.shape)
df_work.head()


Working dataframe shape: (211043, 16)


,proto,service,conn_state,duration,src_bytes,dst_bytes,src_pkts,dst_pkts,missed_bytes,dns_rejected,http_method,http_status_code,ssl_established,weird_notice,label,type
0,tcp,-,OTH,290.371539,101568,2592,108,31,0,-,-,0,-,-,1,backdoor
1,tcp,-,REJ,0.000102,0,0,1,1,0,-,-,0,-,-,1,backdoor
2,tcp,-,REJ,0.000148,0,0,1,1,0,-,-,0,-,-,1,backdoor
3,tcp,-,REJ,0.000113,0,0,1,1,0,-,-,0,-,-,1,backdoor
4,tcp,-,REJ,0.000130,0,0,1,1,0,-,-,0,-,-,1,backdoor


In [4]:
# Inspect unique values in key categorical columns
cat_cols = [
    'proto',
    'service',
    'conn_state',
    'dns_rejected',
    'http_method',
    'ssl_established',
    'weird_notice'
]

for col in cat_cols:
    print(f"\nColumn: {col}")
    print(df_work[col].value_counts().head(10))



Column: proto
proto
tcp     168747
udp      42015
icmp       281
Name: count, dtype: int64

Column: service
service
-             132032
dns            39446
http           37029
ftp             1065
ssl             1025
gssapi           184
dce_rpc          136
smb              108
smb;gssapi        18
Name: count, dtype: int64

Column: conn_state
conn_state
S0       51937
SF       50210
REJ      44852
OTH      23332
SH       12014
S1       10771
S3        6557
SHR       5629
RSTR      1989
RSTRH     1690
Name: count, dtype: int64

Column: dns_rejected
dns_rejected
-    176030
F     28873
T      6140
Name: count, dtype: int64

Column: http_method
http_method
-       210756
GET        266
POST        17
HEAD         4
Name: count, dtype: int64

Column: ssl_established
ssl_established
-    210642
F       272
T       129
Name: count, dtype: int64

Column: weird_notice
weird_notice
-    210687
F       356
Name: count, dtype: int64


In [5]:
# Make a copy to avoid accidental overwrite
df_norm = df_work.copy()

# Replace '-' with 'none' for categorical clarity
replace_dash_cols = [
    'service',
    'dns_rejected',
    'http_method',
    'ssl_established',
    'weird_notice'
]

for col in replace_dash_cols:
    df_norm[col] = df_norm[col].replace('-', 'none')

# Normalize boolean-like columns
bool_map = {
    'T': 'true',
    'F': 'false',
    'none': 'none'
}

for col in ['dns_rejected', 'ssl_established', 'weird_notice']:
    df_norm[col] = df_norm[col].map(bool_map)

# Normalize text columns to lowercase strings
text_cols = ['proto', 'service', 'conn_state', 'http_method']
for col in text_cols:
    df_norm[col] = df_norm[col].astype(str).str.lower()

print("Normalized dataframe shape:", df_norm.shape)
df_norm.head()


Normalized dataframe shape: (211043, 16)


,proto,service,conn_state,duration,src_bytes,dst_bytes,src_pkts,dst_pkts,missed_bytes,dns_rejected,http_method,http_status_code,ssl_established,weird_notice,label,type
0,tcp,none,oth,290.371539,101568,2592,108,31,0,none,none,0,none,none,1,backdoor
1,tcp,none,rej,0.000102,0,0,1,1,0,none,none,0,none,none,1,backdoor
2,tcp,none,rej,0.000148,0,0,1,1,0,none,none,0,none,none,1,backdoor
3,tcp,none,rej,0.000113,0,0,1,1,0,none,none,0,none,none,1,backdoor
4,tcp,none,rej,0.000130,0,0,1,1,0,none,none,0,none,none,1,backdoor


In [6]:
# Helper function for binning
def bin_numeric(x, low, high):
    if x < low:
        return "low"
    elif x < high:
        return "medium"
    else:
        return "high"

# Discretize numeric columns
df_norm['duration_bin'] = df_norm['duration'].apply(lambda x: bin_numeric(x, 1, 10))
df_norm['src_bytes_bin'] = df_norm['src_bytes'].apply(lambda x: bin_numeric(x, 500, 2000))
df_norm['dst_bytes_bin'] = df_norm['dst_bytes'].apply(lambda x: bin_numeric(x, 500, 2000))
df_norm['src_pkts_bin'] = df_norm['src_pkts'].apply(lambda x: bin_numeric(x, 10, 100))
df_norm['dst_pkts_bin'] = df_norm['dst_pkts'].apply(lambda x: bin_numeric(x, 10, 100))
df_norm['missed_bytes_bin'] = df_norm['missed_bytes'].apply(lambda x: bin_numeric(x, 1, 10))

# Check the transformed dataframe
bin_cols = [
    'duration_bin',
    'src_bytes_bin',
    'dst_bytes_bin',
    'src_pkts_bin',
    'dst_pkts_bin',
    'missed_bytes_bin'
]

print("Binned columns preview:")
df_norm[bin_cols + ['label', 'type']].head()


Binned columns preview:


,duration_bin,src_bytes_bin,dst_bytes_bin,src_pkts_bin,dst_pkts_bin,missed_bytes_bin,label,type
0,high,high,high,high,medium,low,1,backdoor
1,low,low,low,low,low,low,1,backdoor
2,low,low,low,low,low,low,1,backdoor
3,low,low,low,low,low,low,1,backdoor
4,low,low,low,low,low,low,1,backdoor


In [7]:
# Drop raw numeric columns
raw_numeric_cols = [
    'duration',
    'src_bytes',
    'dst_bytes',
    'src_pkts',
    'dst_pkts',
    'missed_bytes'
]

df_norm = df_norm.drop(columns=raw_numeric_cols)

print("Final structured dataframe shape:", df_norm.shape)
df_norm.head()


Final structured dataframe shape: (211043, 16)


,proto,service,conn_state,dns_rejected,http_method,http_status_code,ssl_established,weird_notice,label,type,duration_bin,src_bytes_bin,dst_bytes_bin,src_pkts_bin,dst_pkts_bin,missed_bytes_bin
0,tcp,none,oth,none,none,0,none,none,1,backdoor,high,high,high,high,medium,low
1,tcp,none,rej,none,none,0,none,none,1,backdoor,low,low,low,low,low,low
2,tcp,none,rej,none,none,0,none,none,1,backdoor,low,low,low,low,low,low
3,tcp,none,rej,none,none,0,none,none,1,backdoor,low,low,low,low,low,low
4,tcp,none,rej,none,none,0,none,none,1,backdoor,low,low,low,low,low,low


In [8]:
def row_to_text(row):
    tokens = []

    # protocol & service
    tokens.append(f"{row['proto']} protocol")
    if row['service'] != 'none':
        tokens.append(f"{row['service']} service")

    # connection state
    tokens.append(f"connection state {row['conn_state']}")

    # numeric behavior (binned)
    tokens.append(f"{row['duration_bin']} duration")
    tokens.append(f"{row['src_bytes_bin']} outgoing bytes")
    tokens.append(f"{row['dst_bytes_bin']} incoming bytes")
    tokens.append(f"{row['src_pkts_bin']} source packets")
    tokens.append(f"{row['dst_pkts_bin']} destination packets")

    if row['missed_bytes_bin'] != 'low':
        tokens.append(f"{row['missed_bytes_bin']} missed bytes")

    # DNS
    if row['dns_rejected'] == 'true':
        tokens.append("dns rejected")

    # HTTP
    if row['http_method'] != 'none':
        tokens.append(f"http {row['http_method']} request")
        if row['http_status_code'] != 0:
            tokens.append(f"http status {int(row['http_status_code'])}")

    # SSL
    if row['ssl_established'] == 'true':
        tokens.append("ssl established")

    # Weird / anomaly indicator
    if row['weird_notice'] == 'false':
        tokens.append("weird activity detected")

    return " ".join(tokens)


# Create NLP text dataframe
df_text = df_norm.copy()
df_text['text'] = df_text.apply(row_to_text, axis=1)

# Keep only NLP-relevant columns
df_text = df_text[['text', 'label', 'type']]

print("NLP dataset shape:", df_text.shape)
df_text[['text', 'label', 'type']].head()


NLP dataset shape: (211043, 3)


,text,label,type
0,tcp protocol connection state oth high duratio...,1,backdoor
1,tcp protocol connection state rej low duration...,1,backdoor
2,tcp protocol connection state rej low duration...,1,backdoor
3,tcp protocol connection state rej low duration...,1,backdoor
4,tcp protocol connection state rej low duration...,1,backdoor


In [9]:
from sklearn.model_selection import train_test_split

X = df_text['text']
y = df_text['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size :", X_test.shape)

print("\nTrain label distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest label distribution:")
print(y_test.value_counts(normalize=True))


Train size: (168834,)
Test size : (42209,)

Train label distribution:
label
1    0.763081
0    0.236919
Name: proportion, dtype: float64

Test label distribution:
label
1    0.763084
0    0.236916
Name: proportion, dtype: float64


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape :", X_test_tfidf.shape)


TF-IDF train shape: (168834, 153)
TF-IDF test shape : (42209, 153)


In [11]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # important due to imbalance
    n_jobs=-1,
    random_state=42
)

lr.fit(X_train_tfidf, y_train)

print("Logistic Regression training completed.")


Logistic Regression training completed.


In [12]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Predictions
y_pred = lr.predict(X_test_tfidf)
y_prob = lr.predict_proba(X_test_tfidf)[:, 1]

print("Classification Report:\n")
print(classification_report(y_test, y_pred, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_prob))


Classification Report:

              precision    recall  f1-score   support

           0     0.7046    0.9840    0.8212     10000
           1     0.9943    0.8719    0.9291     32209

    accuracy                         0.8985     42209
   macro avg     0.8495    0.9279    0.8751     42209
weighted avg     0.9257    0.8985    0.9035     42209


Confusion Matrix:
[[ 9840   160]
 [ 4126 28083]]

ROC-AUC Score:
0.9820858548852807


In [13]:
# Save the NLP-ready dataset
output_path = "TON_IoT_NLP_Dataset.csv"

df_text.to_csv(output_path, index=False)

print(f"NLP dataset saved successfully as: {output_path}")
print("Shape:", df_text.shape)


NLP dataset saved successfully as: TON_IoT_NLP_Dataset.csv
Shape: (211043, 3)
